In [ ]:
import os, sys, shutil, subprocess
from multiprocessing import cpu_count

print("Python:", sys.version)
print("CPU cores:", cpu_count())

# Kaggle paden
BASE = "/kaggle/working"
APPLIO_DIR = f"{BASE}/Applio"
LOGS_DIR = f"{APPLIO_DIR}/logs"

In [ ]:
APPLIO_REF = "3.6.2"   # probeer bv "main" of een release-tag als dat beter werkt

%cd /kaggle/working
if not os.path.isdir(APPLIO_DIR):
    !git clone https://github.com/IAHispano/Applio --branch {APPLIO_REF} --single-branch

# System deps (portaudio is vaak nodig)
!apt-get update -y
!apt-get install -y portaudio19-dev psmisc

%cd {APPLIO_DIR}

# Snelle/strakke pip via uv (zoals Applio notebooks doen)
!pip -q install uv
!uv pip install -q -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match --system

# Download Applio prerequisites (models/pretraineds etc.)
!python core.py prerequisites --models "True" --pretraineds_hifigan "True" > /dev/null 2>&1

print("✅ Install klaar")

In [ ]:
!python /kaggle/working/Applio/core.py preprocess \
  --model_name "hitsugi_rvc_v2" \
  --dataset_path "/kaggle/input/datasets/michaeltje26/hitsugi" \
  --sample_rate "40000" \
  --cpu_cores "4" \
  --cut_preprocess "Automatic" \
  --process_effects "False" \
  --noise_reduction "False" \
  --chunk_len "3" \
  --overlap_len "0.3" \
  --normalization_mode "none"

In [ ]:
!python /kaggle/working/Applio/core.py extract \
  --model_name "hitsugi_rvc_v2" \
  --f0_method "rmvpe" \
  --sample_rate "40000" \
  --cpu_cores "4" \
  --gpu "0" \
  --embedder_model "contentvec" \
  --embedder_model_custom "" \
  --include_mutes "2"

In [ ]:
!python /kaggle/working/Applio/core.py index \
  --model_name "hitsugi_rvc_v2" \
  --index_algorithm "Auto"

In [ ]:
!python /kaggle/working/Applio/core.py train \
  --model_name "hitsugi_rvc_v2" \
  --save_every_epoch "5" \
  --save_only_latest "True" \
  --save_every_weights "False" \
  --total_epoch "200" \
  --sample_rate "40000" \
  --batch_size "8" \
  --gpu 0 \
  --pretrained "True" \
  --custom_pretrained "False" \
  --overtraining_detector "False" \
  --cleanup "False" \
  --cache_data_in_gpu "False" \
  --vocoder "HiFi-GAN" \
  --checkpointing "False"

In [ ]:
import os

paths = [
    "/kaggle/working/hitsugi_rvc_v2_inference.zip",
    "/kaggle/working/Applio/logs/hitsugi_rvc_v2/wavs",
    "/kaggle/working/Applio/logs/hitsugi_rvc_v2/raw",
    "/kaggle/working/Applio/logs/hitsugi_rvc_v2/f0",
    "/kaggle/working/Applio/logs/hitsugi_rvc_v2/features",
    "/kaggle/working/Applio/logs/hitsugi_rvc_v2/tensorboard"
]

for p in paths:
    if os.path.exists(p):
        if os.path.isdir(p):
            import shutil
            shutil.rmtree(p)
        else:
            os.remove(p)
        print("Deleted:", p)


In [ ]:
import torch, json
from pathlib import Path

RUN_DIR = Path("/kaggle/working/Applio/logs/hitsugi_rvc_v2")

# pak nieuwste G
g_list = sorted(RUN_DIR.glob("G_*.pth"), key=lambda p: p.stat().st_mtime, reverse=True)
G_PATH = g_list[0]

cfg = json.loads((RUN_DIR / "config.json").read_text())

export = {
    "weight": torch.load(G_PATH, map_location="cpu")["model"],
    "config": {
        "sr": cfg.get("sr", 40000),
        "f0": cfg.get("f0", 1),
        "version": cfg.get("version", "v2")
    }
}

OUT = RUN_DIR / "hitsugi_rvc_v2_export.pth"
torch.save(export, OUT)

print("Exported:", OUT)


In [ ]:
%%bash
cd /kaggle/working

# combineer pth achter index
cat hitsugi_rvc_v2.index hitsugi_rvc_v2_export.pth > combined.index

ls -lh combined.index